# Algorithmic Trading and Quantitative Strategies
## Part 8: Factor Investing — Theory and Python Implementation
**Dr. Ayhan Yuksel, CFA, FDP, FRM, PRM**

Bogazici University, EC581

## Table of Contents

1. Foundations of Factor Investing
2. Multi-Factor Models
3. Factor Construction in Python
4. Factor Mimicking Portfolios
5. Information Coefficient Analysis
6. Performance Evaluation
7. Application: Multi-Factor Strategy
8. Exercises

## 1. Foundations of Factor Investing

### 1.1 From CAPM to Multi-Factor Models

The **Capital Asset Pricing Model (CAPM)** states that the expected excess return of an asset is proportional to its market beta:

$$E[R_i] - R_f = \beta_i (E[R_m] - R_f)$$

where:
- $R_i$ = return of asset $i$
- $R_f$ = risk-free rate
- $R_m$ = market return
- $\beta_i = \frac{\text{Cov}(R_i, R_m)}{\text{Var}(R_m)}$

However, CAPM fails to explain many observed return patterns. This led to **multi-factor models** that include additional risk factors.

### 1.2 Types of Factors

**Macroeconomic factors:**
- GDP growth, inflation, interest rates, credit spreads

**Fundamental (style) factors:**
- Value (P/E, P/B), Size (market cap), Momentum, Quality (ROE, profitability), Low volatility

**Statistical factors:**
- Derived from PCA or factor analysis of return covariance matrix

### 1.3 Why Factor Investing Works

**Risk-based explanations:**
- Value stocks are riskier (distress risk) → higher expected returns as compensation
- Small stocks are less liquid → liquidity premium

**Behavioral explanations:**
- Overreaction/underreaction to news
- Herding behavior
- Disposition effect
- Anchoring to past prices/earnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)

## 2. Multi-Factor Models

### 2.1 Fama-French Three-Factor Model

Fama and French (1993) added two factors to CAPM:

$$R_i - R_f = \alpha_i + \beta_{i,MKT} (R_m - R_f) + \beta_{i,SMB} \cdot SMB + \beta_{i,HML} \cdot HML + \epsilon_i$$

where:
- **SMB** (Small Minus Big): Return spread between small-cap and large-cap stocks
- **HML** (High Minus Low): Return spread between value (high B/M) and growth (low B/M) stocks

### 2.2 Fama-French Five-Factor Model

Fama and French (2015) added:

$$R_i - R_f = \alpha_i + \beta_{MKT} MKT + \beta_{SMB} SMB + \beta_{HML} HML + \beta_{RMW} RMW + \beta_{CMA} CMA + \epsilon_i$$

- **RMW** (Robust Minus Weak): Profitability factor
- **CMA** (Conservative Minus Aggressive): Investment factor

### 2.3 Other Common Factors

| Factor | Description | Long | Short |
|:---|:---|:---|:---|
| Momentum | Past winners outperform losers | 12-1 month winners | 12-1 month losers |
| Low Vol | Low-risk stocks outperform | Low beta/vol | High beta/vol |
| Quality | Profitable firms outperform | High ROE/margins | Low ROE/margins |
| Dividend | High yield outperforms | High dividend | Low dividend |

In [ ]:
import getFamaFrenchFactors as gff
import matplotlib.pyplot as plt

# Fetch Fama-French 5 factors (monthly)
ff5 = gff.famaFrench5Factor(frequency="m")
ff5.set_index("date_ff_factors", inplace=True)

# Fetch Momentum separately
mom = gff.momentumFactor(frequency="m")
mom.set_index("date_ff_factors", inplace=True)

# Combine all 6 factors
df = ff5.join(mom[["MOM"]])

# Plot cumulative returns
factors = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "MOM"]
cumulative = (1 + df[factors]).cumprod()

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
for ax, factor in zip(axes.flat, factors):
    ax.plot(cumulative.index, cumulative[factor], linewidth=1)
    ax.set_title(factor, fontsize=13)
    ax.grid(True, alpha=0.3)

fig.suptitle("Fama-French Factors: Cumulative Returns", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 3. Factor Construction in Python

### 3.1 Data Preparation — Fetching Real Stock Data

We select 30 large-cap US stocks and download 5 years of **monthly** price data. We then construct **monthly time-series panels** for three factor signals:

| Signal | Source | Monthly Construction |
|:---|:---|:---|
| **Momentum (12-1)** | Monthly prices | Cumulative return over months *t-12* to *t-1* (skip most recent month) |
| **Value (Trailing P/E)** | `earnings_dates` → Quarterly reported EPS | Month-end price ÷ TTM Diluted EPS (sum of last 4 reported quarters, forward-filled monthly) |
| **Size (Market Cap)** | Balance sheet → shares outstanding × price | Annual shares outstanding (forward- and back-filled) × month-end close price |

Stocks with insufficient price history or missing financials are discarded.

In [ ]:
# ─── Universe: 30 high market-cap US stocks ──────────────────────────
universe = [
    'AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA', 'META', 'TSLA', 'BRK-B',
    'UNH', 'JNJ', 'JPM', 'V', 'XOM', 'PG', 'MA', 'HD', 'CVX', 'MRK',
    'ABBV', 'LLY', 'PEP', 'KO', 'COST', 'AVGO', 'WMT', 'MCD', 'CSCO',
    'CRM', 'ACN', 'ADBE'
]

end_date = pd.Timestamp.today().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.today() - pd.DateOffset(years=5)).strftime('%Y-%m-%d')

print(f"Downloading monthly prices for {len(universe)} stocks: {start_date} → {end_date}")
raw_prices = yf.download(universe, start=start_date, end=end_date, interval='1mo')['Close']

# Flatten MultiIndex columns if present
if isinstance(raw_prices.columns, pd.MultiIndex):
    raw_prices.columns = raw_prices.columns.droplevel(1)

# Drop stocks with more than 10 % missing monthly observations
threshold = len(raw_prices) * 0.90
prices = raw_prices.dropna(axis=1, thresh=int(threshold))
prices = prices.ffill().dropna(axis=1)

tickers = list(prices.columns)
monthly_index = prices.index
monthly_ret = prices.pct_change()

print(f"Retained {len(tickers)} stocks with clean price data")
print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Months: {len(prices)}")

In [ ]:
# ─── Fetch quarterly EPS and annual shares → monthly P/E & Mkt-Cap ──
#
# P/E:  We use `earnings_dates` which provides ~24 quarters of
#       reported EPS per stock. TTM EPS = rolling sum of 4 quarters,
#       then forward-filled to monthly.
#
# Mcap: Annual balance-sheet shares outstanding, forward-filled and
#       back-filled to monthly, multiplied by month-end price.

pe_panel   = pd.DataFrame(index=monthly_index, columns=tickers, dtype=float)
mcap_panel = pd.DataFrame(index=monthly_index, columns=tickers, dtype=float)

failed = []

for t in tickers:
    try:
        tk = yf.Ticker(t)

        # ── Quarterly reported EPS from earnings_dates ────────────────
        ed = tk.earnings_dates
        past_eps = ed.loc[ed['Reported EPS'].notna(), 'Reported EPS'].sort_index()

        # Normalize earnings-announcement dates to quarter-end timestamps
        past_eps.index = past_eps.index.to_period('Q').to_timestamp('Q')
        past_eps = past_eps.groupby(past_eps.index).last()  # deduplicate

        if len(past_eps) < 4:
            failed.append(t); continue

        # TTM EPS = rolling sum of 4 quarters, then forward-fill monthly
        eps_ttm     = past_eps.rolling(4).sum()
        eps_monthly = eps_ttm.reindex(monthly_index, method='ffill')

        pe = prices[t] / eps_monthly
        pe[eps_monthly <= 0] = np.nan   # negative earnings → undefined P/E

        # ── Shares outstanding from annual balance sheet ──────────────
        bs = tk.balance_sheet
        sh_key = None
        for k in ['Ordinary Shares Number', 'Share Issued']:
            if k in bs.index:
                sh_key = k; break
        if sh_key is None:
            failed.append(t); continue

        shares   = bs.loc[sh_key].dropna().sort_index()
        shares_m = shares.reindex(monthly_index, method='ffill').bfill()
        mcap     = prices[t] * shares_m

        pe_panel[t]   = pe
        mcap_panel[t] = mcap

    except Exception:
        failed.append(t)

# ─── Drop tickers that failed ────────────────────────────────────────
valid_pe   = set(pe_panel.columns[pe_panel.notna().mean() > 0.70])
valid_mcap = set(mcap_panel.columns[mcap_panel.notna().mean() > 0.70])
valid      = sorted(valid_pe & valid_mcap - set(failed))

pe_panel    = pe_panel[valid]
mcap_panel  = mcap_panel[valid]
prices      = prices[valid]
monthly_ret = prices.pct_change()
tickers     = valid

print(f"Stocks with complete monthly factor panels: {len(tickers)}")
if failed:
    print(f"Dropped (insufficient data): {failed}")
print(f"P/E panel:   {pe_panel.shape}  (months × stocks)")
print(f"Mcap panel:  {mcap_panel.shape}")
print(f"\nSample — latest month P/E:")
print(pe_panel.iloc[-1].dropna().round(1))

In [ ]:
# ─── Build price-derived monthly signal panels ───────────────────────
#
# Momentum (12-1 month): cumulative return from t-12 to t-1
momentum_signal = prices.pct_change(11).shift(1)

# Trailing volatility: rolling 6-month std of monthly returns
vol_signal = monthly_ret.rolling(6).std()

print("Monthly signal panels constructed:")
print(f"  Momentum (12-1):  {momentum_signal.shape}")
print(f"  Volatility (6m):  {vol_signal.shape}")
print(f"  P/E (TTM):        {pe_panel.shape}")
print(f"  Market Cap:       {mcap_panel.shape}")

In [ ]:
# ─── Helper functions ────────────────────────────────────────────────
def winsorize(series, limits=(0.01, 0.99)):
    """Winsorize a series at given percentiles."""
    lower = series.quantile(limits[0])
    upper = series.quantile(limits[1])
    return series.clip(lower=lower, upper=upper)

def zscore(series):
    """Standardize to z-score."""
    s = series.dropna()
    if s.std() == 0:
        return s * 0.0
    return (s - s.mean()) / s.std()

def rank_normalize(series):
    """Rank-based normalization to uniform [0, 1]."""
    return series.rank(pct=True)

In [ ]:
# ─── Cross-sectional snapshot: latest month ──────────────────────────
# Show a single-month snapshot for illustration (all factors are
# available as full monthly panels for the actual analysis).

stock_data = pd.DataFrame({
    'market_cap':    mcap_panel.iloc[-1],
    'pe_ratio':      pe_panel.iloc[-1],
    'momentum_12m':  momentum_signal.iloc[-1],
    'volatility':    vol_signal.iloc[-1],
}).dropna()

print("Before winsorization (latest month cross-section):")
print(stock_data.describe().round(3))

for col in ['pe_ratio', 'momentum_12m']:
    stock_data[col] = winsorize(stock_data[col])

for col in ['pe_ratio', 'momentum_12m', 'market_cap']:
    stock_data[f'{col}_z'] = zscore(stock_data[col])

print("\nAfter winsorization + z-score:")
print(stock_data.filter(like='_z').describe().round(3))

### 3.2 Composite Factor Scores

Combine multiple signals into a composite score for each factor:

$$\text{Score}_i = \sum_k w_k \cdot z_{i,k}$$

where $z_{i,k}$ is the z-score of signal $k$ for stock $i$.

Our three factors:
- **Value:** low P/E → negate z-score
- **Size:** small market cap → negate z-score (SMB)
- **Momentum:** high past return → positive z-score

In [ ]:
# Construct composite factor scores from REAL data (latest month)
# Value factor: low P/E is good → negate
stock_data['value_score'] = -stock_data['pe_ratio_z']

# Size factor: small cap is good (SMB) → negate
stock_data['size_score'] = -stock_data['market_cap_z']

# Momentum factor: high past return is good
stock_data['momentum_score'] = stock_data['momentum_12m_z']

# Combined alpha score (equal weight)
stock_data['alpha_score'] = (
    stock_data['value_score'] + 
    stock_data['size_score'] + 
    stock_data['momentum_score']
) / 3

# Rank stocks
stock_data['alpha_rank'] = stock_data['alpha_score'].rank(ascending=False)

print("Top 10 stocks by Alpha Score (latest month):")
print(stock_data.nlargest(10, 'alpha_score')[
    ['market_cap', 'pe_ratio', 'momentum_12m', 'alpha_score', 'alpha_rank']
].round(3))

## 4. Factor Mimicking Portfolios (FMP)

A **Factor Mimicking Portfolio** is a long-short portfolio designed to capture the return of a specific factor.

### 4.1 Portfolio Sort Approach (Quintile Portfolios)

1. Sort stocks by factor score
2. Divide into quintiles (5 groups)
3. Long the top quintile, short the bottom quintile
4. The spread (Q5 - Q1) is the factor return

We construct three FMPs — **Momentum**, **Value (inverse P/E)**, and **Size (inverse market cap)** — each re-sorted every month using the monthly signal panels.

In [ ]:
def build_quintile_fmp(signal_panel, returns_panel, n_quintiles=5, min_stocks=10):
    """Build a quintile-sorted long-short factor portfolio.
    
    Parameters
    ----------
    signal_panel : DataFrame (T × N)  — lagged factor signal
    returns_panel : DataFrame (T × N) — next-month returns
    
    Returns
    -------
    DataFrame with columns Q1..Q5 and L/S, indexed by date
    """
    records = []
    for i in range(len(returns_panel)):
        sig = signal_panel.iloc[i].dropna()
        ret = returns_panel.iloc[i].dropna()
        common = sig.index.intersection(ret.index)
        if len(common) < min_stocks:
            continue
        sig, ret = sig[common], ret[common]
        try:
            q = pd.qcut(sig, n_quintiles, labels=range(1, n_quintiles + 1))
        except ValueError:
            continue
        df_m = pd.DataFrame({'sig': sig, 'ret': ret, 'q': q})
        q_ret = df_m.groupby('q')['ret'].mean()
        if len(q_ret) < n_quintiles:
            continue
        records.append({
            'date': returns_panel.index[i],
            'Q1': q_ret.iloc[0], 'Q2': q_ret.iloc[1], 'Q3': q_ret.iloc[2],
            'Q4': q_ret.iloc[3], 'Q5': q_ret.iloc[4],
            'L/S': q_ret.iloc[-1] - q_ret.iloc[0],
        })
    return pd.DataFrame(records).set_index('date')

In [ ]:
# ─── Momentum FMP ────────────────────────────────────────────────────
# Signal is already lagged (shift(1) built into momentum_signal)
fmp_mom = build_quintile_fmp(
    momentum_signal.iloc[12:],   # need 12 months of history
    monthly_ret.iloc[12:]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg_ret = fmp_mom[['Q1','Q2','Q3','Q4','Q5']].mean() * 12
axes[0].bar(range(5), avg_ret, color=['red','salmon','grey','lightgreen','green'])
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(['Q1\n(Low Mom)', 'Q2', 'Q3', 'Q4', 'Q5\n(High Mom)'])
axes[0].set_ylabel('Annualized Return')
axes[0].set_title('Momentum Factor — Quintile Returns')
axes[0].axhline(y=0, color='k', linewidth=0.5)

cum_ls = (1 + fmp_mom['L/S']).cumprod()
axes[1].plot(cum_ls, 'b-', linewidth=2)
axes[1].set_title('Cumulative Long-Short Momentum Return')
axes[1].set_ylabel('Growth of $1')
axes[1].axhline(y=1, color='k', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

ls = fmp_mom['L/S']
t_s, p_v = stats.ttest_1samp(ls, 0)
print(f"Momentum Factor (Real Data):")
print(f"  Ann. Return:     {ls.mean()*12:.2%}")
print(f"  Ann. Volatility: {ls.std()*np.sqrt(12):.2%}")
print(f"  Sharpe Ratio:    {ls.mean()/ls.std()*np.sqrt(12):.3f}")
print(f"  t-statistic:     {t_s:.3f}")
print(f"  p-value:         {p_v:.4f}")

In [ ]:
# ─── Value FMP (inverse P/E: low P/E = high score) ──────────────────
# Lag the P/E signal by 1 month so we sort on last month's P/E
value_signal = -pe_panel.shift(1)  # negate: low P/E → high score

fmp_val = build_quintile_fmp(
    value_signal.iloc[1:],
    monthly_ret.iloc[1:]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg_val = fmp_val[['Q1','Q2','Q3','Q4','Q5']].mean() * 12
axes[0].bar(range(5), avg_val, color=['red','salmon','grey','lightgreen','green'])
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(['Q1\n(High P/E)', 'Q2', 'Q3', 'Q4', 'Q5\n(Low P/E)'])
axes[0].set_ylabel('Annualized Return')
axes[0].set_title('Value Factor (Inverse P/E) — Quintile Returns')
axes[0].axhline(y=0, color='k', linewidth=0.5)

cum_val = (1 + fmp_val['L/S']).cumprod()
axes[1].plot(cum_val, 'r-', linewidth=2)
axes[1].set_title('Cumulative Long-Short Value Return')
axes[1].set_ylabel('Growth of $1')
axes[1].axhline(y=1, color='k', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

ls_v = fmp_val['L/S']
t_s, p_v = stats.ttest_1samp(ls_v, 0)
print(f"Value Factor (Real Monthly P/E):")
print(f"  Ann. Return:     {ls_v.mean()*12:.2%}")
print(f"  Ann. Volatility: {ls_v.std()*np.sqrt(12):.2%}")
print(f"  Sharpe Ratio:    {ls_v.mean()/ls_v.std()*np.sqrt(12):.3f}")
print(f"  t-statistic:     {t_s:.3f}")
print(f"  p-value:         {p_v:.4f}")

In [ ]:
# ─── Size FMP (small minus big: low mcap = high score) ──────────────
size_signal = -mcap_panel.shift(1)  # negate: small cap → high score

fmp_size = build_quintile_fmp(
    size_signal.iloc[1:],
    monthly_ret.iloc[1:]
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg_sz = fmp_size[['Q1','Q2','Q3','Q4','Q5']].mean() * 12
axes[0].bar(range(5), avg_sz, color=['red','salmon','grey','lightgreen','green'])
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(['Q1\n(Large Cap)', 'Q2', 'Q3', 'Q4', 'Q5\n(Small Cap)'])
axes[0].set_ylabel('Annualized Return')
axes[0].set_title('Size Factor (SMB) — Quintile Returns')
axes[0].axhline(y=0, color='k', linewidth=0.5)

cum_sz = (1 + fmp_size['L/S']).cumprod()
axes[1].plot(cum_sz, 'g-', linewidth=2)
axes[1].set_title('Cumulative Long-Short Size (SMB) Return')
axes[1].set_ylabel('Growth of $1')
axes[1].axhline(y=1, color='k', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

ls_s = fmp_size['L/S']
t_s2, p_v2 = stats.ttest_1samp(ls_s, 0)
print(f"Size Factor (Real Monthly Mcap):")
print(f"  Ann. Return:     {ls_s.mean()*12:.2%}")
print(f"  Ann. Volatility: {ls_s.std()*np.sqrt(12):.2%}")
print(f"  Sharpe Ratio:    {ls_s.mean()/ls_s.std()*np.sqrt(12):.3f}")
print(f"  t-statistic:     {t_s2:.3f}")
print(f"  p-value:         {p_v2:.4f}")

### 4.2 Cross-Sectional Regression (Fama-MacBeth)

The Fama-MacBeth (1973) method:

1. **Each period:** Run a cross-sectional regression of returns on factor exposures
$$R_{i,t} = \gamma_{0,t} + \gamma_{1,t} X_{i,t-1} + \epsilon_{i,t}$$

2. **Across time:** The factor premium is the time-series average of $\gamma_{1,t}$
$$\hat{\lambda} = \frac{1}{T} \sum_{t=1}^{T} \hat{\gamma}_{1,t}$$

3. **Test significance:** Using the standard error of $\hat{\gamma}_{1,t}$ across time

We use **three** cross-sectional factor exposures — all recomputed monthly:
- Momentum z-score (12-1 month return)
- Value z-score (inverse trailing P/E)
- Size z-score (inverse market cap)

In [ ]:
def fama_macbeth(returns_panel, factors_panel):
    """
    Fama-MacBeth cross-sectional regression.
    
    Parameters:
    -----------
    returns_panel : pd.DataFrame (T x N) - Asset returns
    factors_panel : dict of pd.DataFrame (T x N) - Factor exposures (lagged)
    
    Returns:
    --------
    pd.DataFrame with factor premia estimates and t-statistics
    """
    T = len(returns_panel)
    factor_names = list(factors_panel.keys())
    gammas = {f: [] for f in factor_names}
    gammas['intercept'] = []
    
    for t in range(T):
        ret = returns_panel.iloc[t].dropna()
        X = pd.DataFrame({f: factors_panel[f].iloc[t] for f in factor_names})
        X = X.loc[ret.index].dropna()
        ret = ret.loc[X.index]
        
        if len(ret) < 10:
            continue
        
        X_const = sm.add_constant(X)
        model = sm.OLS(ret, X_const).fit()
        
        gammas['intercept'].append(model.params['const'])
        for f in factor_names:
            gammas[f].append(model.params[f])
    
    results = {}
    for name in ['intercept'] + factor_names:
        g = np.array(gammas[name])
        if len(g) == 0:
            continue
        mean_g = g.mean()
        se_g = g.std() / np.sqrt(len(g))
        t_stat = mean_g / se_g if se_g > 0 else 0
        results[name] = {
            'premium': mean_g,
            'std_error': se_g,
            't_stat': t_stat,
            'p_value': 2 * (1 - stats.t.cdf(abs(t_stat), len(g)-1)),
        }
    
    return pd.DataFrame(results).T

# ─── Build monthly z-scored factor exposure panels (lagged) ──────────
mom_z   = momentum_signal.apply(zscore, axis=1).shift(1)
value_z = (-pe_panel).apply(zscore, axis=1).shift(1)      # low P/E → high
size_z  = (-mcap_panel).apply(zscore, axis=1).shift(1)    # small cap → high

# Align dates
common_idx    = monthly_ret.index[13:]   # 12 months for mom + 1 lag
ret_panel     = monthly_ret.loc[common_idx, tickers]
mom_panel_fm  = mom_z.reindex(common_idx)[tickers]
val_panel_fm  = value_z.reindex(common_idx)[tickers]
size_panel_fm = size_z.reindex(common_idx)[tickers]

fm_results = fama_macbeth(
    ret_panel,
    {'momentum': mom_panel_fm, 'value': val_panel_fm, 'size': size_panel_fm}
)
print("Fama-MacBeth Results (Real Monthly Data — 3 Factors):")
print(fm_results.round(4))

### 4.3 Statistical Factor Approach (PCA)

Principal Component Analysis (PCA) extracts latent factors from the covariance matrix of returns.

The first few principal components capture the dominant return patterns (market, sector, style).

In [ ]:
# PCA on real monthly stock returns
ret_for_pca = monthly_ret.loc[common_idx, tickers].dropna()

scaler = StandardScaler()
returns_scaled = scaler.fit_transform(ret_for_pca)

n_comp = min(10, len(tickers))
pca = PCA(n_components=n_comp)
pca.fit(returns_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, n_comp + 1), pca.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('PCA: Variance Explained by Component (Real Returns)')

cum_var = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, n_comp + 1), cum_var, 'bo-')
axes[1].axhline(y=0.8, color='r', linestyle='--', label='80% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('PCA: Cumulative Variance Explained')
axes[1].legend()

plt.tight_layout()
plt.show()

pc_returns = pca.transform(returns_scaled)[:, :3]
pc_df = pd.DataFrame(pc_returns, index=ret_for_pca.index, columns=['PC1', 'PC2', 'PC3'])

print(f"\nFirst 3 PCs explain {cum_var[2]:.1%} of variance")
print(f"\nPC Factor Return Statistics:")
print(pc_df.describe().round(4))

## 5. Information Coefficient Analysis

The **Information Coefficient (IC)** measures the predictive power of a factor signal. It is the cross-sectional correlation between the factor signal and subsequent returns:

$$IC_t = \text{Corr}(\text{Signal}_{t-1}, R_t)$$

- **IC > 0:** Factor has predictive power
- **IC ~ 0.05:** Typical for good factors
- **IC > 0.1:** Exceptional
- **IR (Information Ratio):** $IR = \frac{\overline{IC}}{\sigma(IC)}$ — IC adjusted for consistency

We compute IC for all three factors — **Momentum**, **Value (1/PE)**, and **Size (1/Mcap)** — each from monthly panels.

In [ ]:
# ─── IC for momentum, value, and size signals ────────────────────────
def compute_ic_series(signal_panel, returns_panel, min_stocks=10):
    """Compute monthly IC between lagged signal and next-month return."""
    ic_vals, ic_dates = [], []
    for i in range(1, len(returns_panel)):
        sig = signal_panel.iloc[i - 1].dropna()   # lagged signal
        ret = returns_panel.iloc[i].dropna()
        common = sig.index.intersection(ret.index)
        if len(common) < min_stocks:
            continue
        ic = np.corrcoef(sig[common], ret[common])[0, 1]
        if np.isnan(ic):
            continue
        ic_vals.append(ic)
        ic_dates.append(returns_panel.index[i])
    return pd.Series(ic_vals, index=ic_dates)

ic_mom  = compute_ic_series(momentum_signal, monthly_ret)
ic_val  = compute_ic_series(-pe_panel, monthly_ret)      # low P/E = good
ic_size = compute_ic_series(-mcap_panel, monthly_ret)    # small cap = good

for name, ic in [('Momentum', ic_mom), ('Value (1/PE)', ic_val), ('Size (1/Mcap)', ic_size)]:
    ir = ic.mean() / ic.std() if ic.std() > 0 else 0
    t_s, p_v = stats.ttest_1samp(ic, 0)
    print(f"{name:18s}  Mean IC={ic.mean():.4f}  Std={ic.std():.4f}  "
          f"IC>0={100*(ic>0).mean():.0f}%  IR={ir:.3f}  t={t_s:.2f}  p={p_v:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (name, ic) in zip(axes, [('Momentum', ic_mom), ('Value (1/PE)', ic_val),
                                   ('Size (1/Mcap)', ic_size)]):
    ax.bar(range(len(ic)), ic,
           color=['green' if x > 0 else 'red' for x in ic], alpha=0.7)
    ax.axhline(y=ic.mean(), color='blue', linestyle='--',
               label=f'Mean IC = {ic.mean():.3f}')
    ax.set_title(f'{name} — Monthly IC')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Performance Evaluation of Factor Portfolios

Key metrics for evaluating factor portfolios:

In [ ]:
def evaluate_factor_portfolio(returns, name="Factor"):
    """Comprehensive evaluation of a return series."""
    ann_ret = returns.mean() * 12
    ann_vol = returns.std() * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    
    cum = (1 + returns).cumprod()
    dd = cum / cum.cummax() - 1
    max_dd = dd.min()
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
    
    t_stat, p_val = stats.ttest_1samp(returns, 0)
    skew = returns.skew()
    kurt = returns.kurtosis()
    
    print(f"{'='*40}")
    print(f"{name} Portfolio Performance")
    print(f"{'='*40}")
    print(f"Ann. Return:    {ann_ret:.2%}")
    print(f"Ann. Volatility:{ann_vol:.2%}")
    print(f"Sharpe Ratio:   {sharpe:.3f}")
    print(f"Max Drawdown:   {max_dd:.2%}")
    print(f"Calmar Ratio:   {calmar:.3f}")
    print(f"Skewness:       {skew:.3f}")
    print(f"Excess Kurtosis:{kurt:.3f}")
    print(f"t-statistic:    {t_stat:.3f}")
    print(f"p-value:        {p_val:.4f}")
    print(f"% Positive:     {(returns > 0).mean():.1%}")
    
    return {'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe, 
            'max_dd': max_dd, 't_stat': t_stat}

evaluate_factor_portfolio(fmp_mom['L/S'], "Momentum L/S")
print()
evaluate_factor_portfolio(fmp_val['L/S'], "Value L/S (monthly P/E)")
print()
evaluate_factor_portfolio(fmp_size['L/S'], "Size L/S (monthly Mcap)")

## 7. Application: Multi-Factor Strategy with Real Data

We construct a **multi-factor long-short portfolio** combining all three signals — Momentum, Value (inverse trailing P/E), and Size (inverse market cap). **All three signals are recomputed from their monthly panels each month** — no static snapshots.

In [ ]:
# ─── Monthly multi-factor L/S strategy ───────────────────────────────
multi_factor_returns = []

for i in range(12, len(monthly_ret)):
    # Lagged signals
    mom_sig  = momentum_signal.iloc[i].dropna()
    pe_sig   = pe_panel.iloc[i - 1].dropna()
    mcap_sig = mcap_panel.iloc[i - 1].dropna()
    ret      = monthly_ret.iloc[i].dropna()

    # Common stocks with all signals and return
    common = (mom_sig.index
              .intersection(pe_sig.index)
              .intersection(mcap_sig.index)
              .intersection(ret.index))
    if len(common) < 10:
        continue

    # Cross-sectional z-scores this month
    z_mom  = zscore(mom_sig[common])
    z_val  = zscore(-pe_sig[common])     # low P/E → high score
    z_size = zscore(-mcap_sig[common])   # small cap → high score

    composite = (z_mom + z_val + z_size) / 3

    n = len(common)
    top    = composite.nlargest(n // 5).index
    bottom = composite.nsmallest(n // 5).index

    long_ret  = ret[top].mean()
    short_ret = ret[bottom].mean()

    multi_factor_returns.append({
        'date':  monthly_ret.index[i],
        'long':  long_ret,
        'short': short_ret,
        'L/S':   long_ret - short_ret,
    })

mf_df = pd.DataFrame(multi_factor_returns).set_index('date')

evaluate_factor_portfolio(mf_df['L/S'],
                          "Multi-Factor L/S (Momentum + Value + Size)")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
cum = (1 + mf_df[['long', 'short', 'L/S']]).cumprod()
for col in cum.columns:
    axes[0].plot(cum[col], label=col, linewidth=1.5)
axes[0].set_title('Multi-Factor Strategy: Cumulative Returns (Real Data)')
axes[0].legend()
axes[0].axhline(y=1, color='k', linestyle='--', linewidth=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(mf_df)), mf_df['L/S'],
           color=['green' if x > 0 else 'red' for x in mf_df['L/S']], alpha=0.7)
axes[1].set_title('Monthly L/S Returns')
axes[1].set_ylabel('Return')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Compare individual factors vs. multi-factor ─────────────────────
comparison = pd.DataFrame({
    'Momentum L/S': fmp_mom['L/S'],
    'Value L/S':    fmp_val['L/S'],
    'Size L/S':     fmp_size['L/S'],
    'Multi-Factor L/S': mf_df['L/S'],
}).dropna()

cum_comp = (1 + comparison).cumprod()

fig, ax = plt.subplots(figsize=(14, 5))
for col in cum_comp.columns:
    ax.plot(cum_comp[col], label=col, linewidth=2)
ax.set_title('Factor Strategy Comparison (Real Data — All Monthly Signals)')
ax.legend()
ax.axhline(y=1, color='k', linestyle='--', linewidth=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for col in comparison.columns:
    print()
    evaluate_factor_portfolio(comparison[col], col)

## 8. Exercises

1. **Value Factor**: Experiment with alternative value signals — e.g., price-to-sales or earnings yield (E/P). Do they produce a stronger or weaker value premium than trailing P/E?

2. **Low Volatility Factor**: Sort stocks by their trailing 6-month volatility. Construct a long-short portfolio (long low-vol, short high-vol). Does the low volatility anomaly appear?

3. **Multi-Factor Model**: Experiment with different weighting schemes (e.g., 50% momentum, 30% value, 20% size). How does the Sharpe ratio change?

4. **Fama-MacBeth Regression**: Add the volatility signal as a fourth factor in the Fama-MacBeth regression. Is the low-volatility premium significant after controlling for momentum, value, and size?

---
### References
- Fama, E. F., & French, K. R. (1993). *Common Risk Factors in the Returns on Stocks and Bonds.* Journal of Financial Economics.
- Fama, E. F., & French, K. R. (2015). *A Five-Factor Asset Pricing Model.* Journal of Financial Economics.
- Fama, E. F., & MacBeth, J. D. (1973). *Risk, Return, and Equilibrium: Empirical Tests.* Journal of Political Economy.
- Grinold, R. & Kahn, R. (2000). *Active Portfolio Management.* McGraw-Hill.
- Fabozzi, Focardi, & Kolm (2010). *Quantitative Equity Investing.* Wiley.
- Bali, Engle & Murray (2016). *Empirical Asset Pricing: The Cross Section of Stock Returns.* Wiley.